# Final Results, Discussion, Limitations, and Conclusion

This notebook consolidates the complete project findings after data preparation, exploratory analysis, machine learning preprocessing, baseline modeling, hyperparameter tuning, SHAP explainability, probability calibration, and threshold optimization.

It does not train new models. Its purpose is to organize the existing findings into a clear research-style interpretation suitable for a report, presentation, GitHub repository, or paper draft.

## 1. Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load Saved Result Tables

The notebook attempts to load outputs saved by the earlier phases. If a file is unavailable, the markdown interpretations and manually recorded project results remain usable.

In [2]:
possible_artifact_paths = [Path("../artifacts"), Path("artifacts")]
artifact_path = next((p for p in possible_artifact_paths if p.exists()), None)

if artifact_path is None:
    print("Artifacts folder not found. Markdown sections can still be used.")
else:
    print("Artifacts folder:", artifact_path.resolve())


def load_csv_if_available(path, **kwargs):
    if path.exists():
        return pd.read_csv(path, **kwargs)
    print("Not found:", path)
    return None

baseline_results = tuned_results = baseline_vs_tuned = None
cv_overfitting = shap_original_importance = None
calibration_results = candidate_thresholds = None
threshold_comparison = recommended_threshold_summary = None

if artifact_path is not None:
    baseline_results = load_csv_if_available(
        artifact_path / "baseline_models" / "baseline_model_results.csv",
        index_col=0
    )
    tuned_results = load_csv_if_available(
        artifact_path / "tuned_models" / "tuned_model_results.csv",
        index_col=0
    )
    baseline_vs_tuned = load_csv_if_available(
        artifact_path / "tuned_models" / "baseline_vs_tuned_comparison.csv",
        index_col=0
    )
    cv_overfitting = load_csv_if_available(
        artifact_path / "tuned_models" / "cv_overfitting_summary.csv",
        index_col=0
    )
    shap_original_importance = load_csv_if_available(
        artifact_path / "shap_explainability" / "global_original_feature_importance.csv"
    )
    calibration_results = load_csv_if_available(
        artifact_path / "calibration_thresholds" / "calibration_model_comparison.csv",
        index_col=0
    )
    candidate_thresholds = load_csv_if_available(
        artifact_path / "calibration_thresholds" / "candidate_thresholds.csv",
        index_col=0
    )
    threshold_comparison = load_csv_if_available(
        artifact_path / "calibration_thresholds" / "threshold_confusion_comparison.csv",
        index_col=0
    )
    recommended_threshold_summary = load_csv_if_available(
        artifact_path / "calibration_thresholds" / "recommended_threshold_summary.csv"
    )

Artifacts folder: C:\IPD\IPD\artifacts


## 3. Study Population and Outcome Summary

The project used the NFHS-5 Birth Recode dataset and restricted the study population to institutional deliveries. The final modeling dataset contained:

- **201,311 institutional delivery records**
- **158,429 unique respondents**
- **36 predictor variables**
- **22.23% Cesarean deliveries**
- **77.77% vaginal deliveries**

Because Cesarean delivery was the minority class, model evaluation emphasized Recall, F1-score, ROC-AUC, and PR-AUC rather than relying only on Accuracy.

## 4. Baseline Model Results

In [3]:
if baseline_results is not None:
    display(baseline_results.round(4))
else:
    baseline_results_manual = pd.DataFrame({
        "accuracy": [0.7259, 0.7383, 0.7964, 0.7229],
        "precision": [0.4325, 0.4443, 0.5866, 0.3780],
        "recall": [0.7479, 0.7071, 0.2846, 0.3821],
        "f1_score": [0.5481, 0.5457, 0.3833, 0.3800],
        "roc_auc": [0.8035, 0.7943, 0.7904, 0.6012],
        "pr_auc": [0.5333, 0.5109, 0.5040, 0.2818]
    }, index=["XGBoost", "Logistic Regression", "Random Forest", "Decision Tree"])
    display(baseline_results_manual.round(4))

,accuracy,precision,recall,f1_score,roc_auc,pr_auc,training_time_seconds
model,,,,,,,
XGBoost,0.7259,0.4325,0.7479,0.5481,0.8035,0.5333,3.7177
Logistic Regression,0.7383,0.4443,0.7071,0.5457,0.7943,0.5109,6.1139
Random Forest,0.7964,0.5866,0.2846,0.3833,0.7904,0.5040,6.4624
Decision Tree,0.7229,0.3780,0.3821,0.3800,0.6012,0.2818,2.6840


### Baseline Model Interpretation

Four baseline classifiers were evaluated: Logistic Regression, Decision Tree, Random Forest, and XGBoost.

XGBoost achieved the strongest overall baseline performance, with the highest Recall, F1-score, ROC-AUC, and PR-AUC. Logistic Regression remained competitive despite being the simplest model, showing that the selected predictors contained meaningful linear information.

The baseline Random Forest had the highest Accuracy and Precision but detected only 28.46% of actual Cesarean deliveries. This demonstrated why Accuracy alone was misleading in the imbalanced dataset. The Decision Tree produced the weakest discrimination and was not selected for further tuning.

## 5. Hyperparameter-Tuning Results

In [4]:
if tuned_results is not None:
    display(tuned_results.round(4))
else:
    tuned_results_manual = pd.DataFrame({
        "train_accuracy": [0.7352, 0.7650, 0.7393],
        "test_accuracy": [0.7248, 0.7383, 0.7384],
        "precision": [0.4313, 0.4444, 0.4444],
        "recall": [0.7471, 0.7101, 0.7073],
        "f1_score": [0.5469, 0.5467, 0.5459],
        "roc_auc": [0.8033, 0.8002, 0.7943],
        "pr_auc": [0.5330, 0.5262, 0.5108],
        "best_cv_pr_auc": [0.5291, 0.5204, 0.5085]
    }, index=["XGBoost", "Random Forest", "Logistic Regression"])
    display(tuned_results_manual.round(4))

,train_accuracy,test_accuracy,precision,recall,f1_score,roc_auc,pr_auc,best_cv_pr_auc,tuning_time_seconds
model,,,,,,,,,
XGBoost,0.7352,0.7248,0.4313,0.7471,0.5469,0.8033,0.5330,0.5291,45.2635
Random Forest,0.7650,0.7383,0.4444,0.7101,0.5467,0.8002,0.5262,0.5204,457.0531
Logistic Regression,0.7393,0.7384,0.4444,0.7073,0.5459,0.7943,0.5108,0.5085,642.4273


### Hyperparameter-Tuning Interpretation

Hyperparameter tuning affected the models differently.

Logistic Regression changed very little, suggesting that its baseline settings were already close to optimal. Random Forest showed the largest improvement: Recall increased from **28.46% to 71.01%**, along with improvements in F1-score, ROC-AUC, and PR-AUC. XGBoost changed only marginally, indicating that the initial settings were already near optimal.

The tuned XGBoost model remained the strongest overall model because it achieved the highest Recall, F1-score, ROC-AUC, PR-AUC, and cross-validated PR-AUC. It was therefore selected as the final model.

## 6. Overfitting and Generalization

In [5]:
if cv_overfitting is not None:
    display(cv_overfitting.round(4))
else:
    cv_overfitting_manual = pd.DataFrame({
        "mean_train_pr_auc": [0.6266, 0.5739, 0.5094],
        "mean_validation_pr_auc": [0.5204, 0.5291, 0.5085],
        "train_validation_gap": [0.1062, 0.0448, 0.0009]
    }, index=["Random Forest", "XGBoost", "Logistic Regression"])
    display(cv_overfitting_manual.round(4))

,mean_train_pr_auc,mean_validation_pr_auc,train_validation_gap
model,,,
Random Forest,0.6266,0.5204,0.1062
XGBoost,0.5739,0.5291,0.0448
Logistic Regression,0.5094,0.5085,0.0009


### Generalization Interpretation

Logistic Regression showed virtually no train-validation gap, indicating excellent stability. Random Forest showed the largest gap and therefore the greatest tendency to fit training-specific patterns. XGBoost showed only mild overfitting while achieving the highest validation PR-AUC.

The difference between XGBoost training Accuracy (73.52%) and test Accuracy (72.48%) was approximately one percentage point, supporting good generalization to unseen respondents.

## 7. Final Model Performance

The final tuned XGBoost model achieved:

| Metric | Value |
|---|---:|
| Training Accuracy | 0.7352 |
| Test Accuracy | 0.7248 |
| Precision | 0.4313 |
| Recall | 0.7471 |
| F1-score | 0.5469 |
| ROC-AUC | 0.8033 |
| PR-AUC | 0.5330 |
| Best cross-validated PR-AUC | 0.5291 |

The model correctly identified approximately **75 out of every 100 actual Cesarean deliveries**. Precision remained moderate because higher sensitivity also produced more false-positive predictions, reflecting a threshold-dependent trade-off.

## 8. SHAP Global Feature Importance

In [6]:
if shap_original_importance is not None:
    display(shap_original_importance.head(20).round(6))
else:
    shap_importance_manual = pd.DataFrame({
        "original_feature": [
            "facility_type", "total_children_ever_born", "bmi", "wealth_index",
            "age_at_first_birth", "anc_doctor", "education_years", "anc_visits",
            "social_group", "preceding_birth_interval", "swelling_during_pregnancy",
            "anc_anganwadi_worker", "religion", "birth_order", "anc_nurse_midwife",
            "maternal_age", "residence", "twin_order", "anc_asha_worker",
            "health_insurance"
        ],
        "mean_absolute_shap": [
            0.680196, 0.262413, 0.249247, 0.163607, 0.142156, 0.136787,
            0.135703, 0.134337, 0.100929, 0.078568, 0.078259, 0.069499,
            0.060148, 0.058721, 0.051037, 0.047214, 0.044248, 0.041817,
            0.036393, 0.022992
        ]
    })
    display(shap_importance_manual.round(6))

,original_feature,mean_absolute_shap
0,facility_type,0.680196
1,total_children_ever_born,0.262413
2,bmi,0.249247
3,wealth_index,0.163607
4,age_at_first_birth,0.142156
5,anc_doctor,0.136787
6,education_years,0.135703
7,anc_visits,0.134337
8,social_group,0.100929
9,preceding_birth_interval,0.078568


### SHAP Interpretation

Facility type was the most influential predictor by a substantial margin. Private-facility delivery generally pushed predictions toward Cesarean delivery, consistent with the much higher Cesarean rate observed in private facilities during EDA.

Other influential factors included total children ever born, maternal BMI, household wealth, age at first birth, doctor-provided ANC, education, ANC visits, social group, and preceding birth interval.

Higher BMI, greater wealth, later age at first birth, more ANC visits, and private-facility delivery commonly contributed to higher predicted Cesarean probability. These are model-learned associations and must not be interpreted as causal effects.

## 9. SHAP Local Explanations

The highest predicted-risk case had a predicted Cesarean probability of **97.84%** and was correctly classified. Its prediction was mainly driven by high BMI, private-facility delivery, twin-related encoded information, later age at first birth, more ANC visits, and higher socioeconomic indicators.

The lowest predicted-risk case had a predicted Cesarean probability of **1.01%** and was correctly classified as vaginal. Its prediction was mainly reduced by a non-private facility category, low wealth, low BMI, younger age at first birth, no doctor-provided ANC, fewer ANC visits, and younger maternal age.

The false-positive case strongly resembled high-risk Cesarean cases in the training data despite having a vaginal outcome. The false-negative case resembled lower-risk cases despite an actual Cesarean outcome, suggesting that important clinical or emergency factors were not captured in the available predictor set.

## 10. Calibration and Threshold-Optimization Results

In [7]:
if calibration_results is not None:
    print("Calibration model comparison:")
    display(calibration_results.round(4))

if candidate_thresholds is not None:
    print("Candidate thresholds:")
    display(candidate_thresholds.round(4))

if threshold_comparison is not None:
    print("Threshold confusion comparison:")
    display(threshold_comparison.round(4))

if recommended_threshold_summary is not None:
    print("Recommended threshold summary:")
    display(recommended_threshold_summary.round(4))

if all(item is None for item in [
    calibration_results,
    candidate_thresholds,
    threshold_comparison,
    recommended_threshold_summary
]):
    print("Calibration tables were not found. Run Notebook 07 to populate this section.")

Calibration model comparison:


,accuracy,precision,recall,specificity,f1_score,roc_auc,pr_auc,brier_score
model,,,,,,,,
Isotonic,0.8017,0.6087,0.3019,0.9445,0.4036,0.8031,0.5341,0.1361
Sigmoid,0.8022,0.6050,0.3163,0.9410,0.4155,0.8033,0.5339,0.1364
Uncalibrated,0.7248,0.4313,0.7471,0.7185,0.5469,0.8033,0.5330,0.1827


Candidate thresholds:


,threshold,accuracy,precision,recall,specificity,f1_score,youden_j,predicted_positive_rate
strategy,,,,,,,,
Default 0.50,0.50,0.8017,0.6087,0.3019,0.9445,0.4036,0.2465,0.1103
Maximum F1-score,0.29,0.7563,0.4666,0.6737,0.7799,0.5514,0.4536,0.3209
Maximum Youden J,0.22,0.7160,0.4230,0.7629,0.7026,0.5442,0.4655,0.4008
Recall at least 80%,0.19,0.6917,0.4026,0.8004,0.6606,0.5357,0.4610,0.4419


Threshold confusion comparison:


,threshold,true_negative,false_positive,false_negative,true_positive,precision,recall,specificity,f1_score
strategy,,,,,,,,,
Default threshold,0.50,29577,1737,6247,2702,0.6087,0.3019,0.9445,0.4036
Recommended threshold,0.29,24423,6891,2920,6029,0.4666,0.6737,0.7799,0.5514
Recall ≥ 80%,0.19,20685,10629,1786,7163,0.4026,0.8004,0.6606,0.5357


Recommended threshold summary:


,probability_model,recommended_threshold,accuracy,precision,recall,specificity,f1_score,roc_auc,pr_auc,brier_score
0,Isotonic,0.29,0.7563,0.4666,0.6737,0.7799,0.5514,0.8031,0.5341,0.1361


### Calibration Interpretation

The original XGBoost probabilities were compared with sigmoid and isotonic calibration using calibration curves and the Brier score. Additional calibration produced only limited improvement, indicating that the final XGBoost probabilities were already reasonably reliable.

Threshold optimization was still necessary because 0.50 is only a default cutoff. Thresholds were compared using Precision, Recall, Specificity, F1-score, Youden's J statistic, and false-positive and false-negative counts.

The F1-optimal threshold was selected as the main operating threshold because it provided the strongest balance between Precision and Recall. A sensitivity-focused threshold was retained as an alternative when reducing missed Cesarean cases is the main priority.

## 11. Integration of EDA, Modeling, and SHAP Findings

EDA and SHAP were broadly consistent. Facility type showed the strongest categorical association in EDA and was also the most important predictor in the final model. Maternal BMI, wealth, education, ANC visits, doctor-provided ANC, reproductive history, and age at first birth were also important in both descriptive and predictive analyses.

This consistency suggests that the model learned meaningful patterns present in the observed data. However, these patterns may also reflect healthcare access, referral behavior, provider practices, economic incentives, and socioeconomic differences.

## 12. Why XGBoost Performed Best

XGBoost was well suited to this dataset because it can model nonlinear relationships, interactions, complex threshold effects, and class imbalance. Logistic Regression remained competitive because many predictors had broad linear associations. Random Forest improved substantially after tuning but showed a larger train-validation gap.

XGBoost therefore achieved the strongest overall balance of discrimination, sensitivity, PR-AUC, and generalization.

## 13. Clinical and Public-Health Implications

The model shows that Cesarean delivery can be predicted to a meaningful extent using maternal, reproductive, socioeconomic, antenatal-care, and facility-related variables available in NFHS-5.

Potential research applications include population-level risk stratification, identification of high-prediction groups, examination of public-private differences, maternal-health policy analysis, and resource-planning studies.

The model should not currently be used as an independent clinical decision tool because it was developed from survey data and has not undergone external or prospective validation.

## 14. Study Strengths

Key strengths include:

- Large nationally representative maternal-health dataset
- Manual DHS metadata verification
- Careful target-leakage removal
- Restriction to information available before the delivery decision
- Respondent-level train-test splitting
- Grouped cross-validation
- Separate preprocessing for linear and tree-based models
- Imbalance-aware evaluation metrics
- Baseline-versus-tuned model comparison
- SHAP global and local explainability
- Calibration assessment
- Threshold optimization
- Reproducible artifact saving

## 15. Study Limitations


### Previous Cesarean history
Previous Cesarean delivery could not be reliably engineered because Cesarean status was inconsistently recorded for earlier births.

### Limited clinical detail
The dataset lacked detailed fetal distress, placental abnormalities, labour progression, fetal presentation, physician decision-making, hospital protocols, and emergency indication variables.

### Retrospective design
The model identifies associations and predictive patterns rather than causal effects.

### Recall and reporting bias
Some variables were self-reported and may contain recall error or misclassification.

### Facility-type dominance
Facility type may capture case mix, referral patterns, provider practices, economic incentives, or coding differences. It should not be interpreted causally.

### Small other-facility group
The small `other` category showed unusually low Cesarean rates, and strong SHAP effects may partly reflect sparse-data behavior.

### No external or prospective validation
The model has not been tested on another survey, another country, hospital data, or a real clinical workflow.

### Threshold interpretation
The recommended threshold is a statistical operating point, not a clinically approved decision threshold.

## 16. Future Work

Future extensions could include:

- External validation on another maternal-health dataset
- Validation using another NFHS survey round
- State-specific or district-level models
- Separate public and private facility models
- Addition of reliable previous Cesarean history
- Inclusion of richer clinical and fetal variables
- Fairness analysis across socioeconomic and demographic groups
- Survey-weight-aware modeling and evaluation
- External recalibration
- Prospective hospital validation
- Interpretable web-based research prototype
- Comparison with additional gradient-boosting methods
- Sensitivity analysis excluding the small other-facility category
- Comparison with a reduced model excluding facility type

## 17. Final Conclusion

This project developed an end-to-end explainable machine learning pipeline for predicting Cesarean section delivery among institutional births in India using NFHS-5 data.

After variable verification, leakage prevention, EDA, respondent-level splitting, preprocessing, baseline modeling, and tuning, XGBoost was selected as the final model. It achieved Recall of **74.71%**, F1-score of **0.5469**, ROC-AUC of **0.8033**, and PR-AUC of **0.5330**.

SHAP identified facility type, reproductive history, BMI, wealth, age at first birth, education, and ANC utilization as major predictors. Calibration and threshold analysis further assessed probability reliability and operating behavior.

The model should currently be regarded as a research and population-health analysis tool rather than a clinical decision system. External validation, richer clinical data, fairness evaluation, and prospective assessment are required before practical deployment.